<a href="https://colab.research.google.com/github/SasankaPandaSCIT/Automate-with-Gen-AI-Agents/blob/main/Module%204/4.3%20Using%20Tools%20and%20Memory/2%20Tutorial%20-%20Function%20Calling%20Deep%20Dive%20(JSON%20Schemas)%20Openrouter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install pinned dependencies (Colab-ready; safe to re-run).
# Based on latest compatible versions
!pip install -q langchain-core==1.6.3 langchain-classic==1.0.8 langchain-openai==1.6.2 pydantic==2.13.5

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.8/571.8 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 45.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.8/125.8 kB 5.9 MB/s eta 0:00:00


## Tutorial: OpenAI/Gemini Function Calling Deep Dive (w/ JSON Schemas)
We’ll define strict tool schemas with Pydantic and show structured outputs.


In [2]:
import os
from getpass import getpass
from langchain_openai import ChatOpenAI

# Enter your OpenRouter API key securely when prompted.
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY") or getpass("Enter your OpenRouter API key: ")

# OpenRouter provides an OpenAI-compatible API.
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

# You can change this to any compatible OpenRouter model.
MODEL = "openai/gpt-4.1-mini"

llm = ChatOpenAI(
    model=MODEL,
    api_key=OPENROUTER_API_KEY,
    base_url=OPENROUTER_BASE_URL,
    temperature=0,
    seed=42,
)

print("OpenRouter configured successfully.")
print("Model:", MODEL)
from typing import List
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool

# Reuse the OpenRouter-configured model above.


Enter your OpenRouter API key: ··········
OpenRouter configured successfully.
Model: openai/gpt-4.1-mini


### Step 1: Define a schema with Pydantic
We’ll create a `WeatherRequest` and expose a `get_weather` tool with a strict schema.


In [3]:
class WeatherRequest(BaseModel):
    city: str = Field(..., description="City name, e.g., 'San Francisco'")
    unit: str = Field("celsius", description="Temperature unit: 'celsius' or 'fahrenheit'")

@tool("get_weather", args_schema=WeatherRequest, return_direct=True, description="Get the weather in a given city.")
def get_weather(city: str, unit: str = "celsius") -> str:

    # Demo: pretend fetch; return structured string
    sample = {"San Francisco": 18, "New York": 24, "London": 19}
    temp_c = sample.get(city, 20)
    if unit == "fahrenheit":
        temp = round((temp_c * 9/5) + 32)
        return f"{{\"city\": \"{city}\", \"temp\": {temp}, \"unit\": \"F\"}}"
    return f"{{\"city\": \"{city}\", \"temp\": {temp_c}, \"unit\": \"C\"}}"


### Step 2: Show the generated JSON schema
Pydantic provides JSON schema; most providers accept the same shape for function calling.


In [4]:
from pprint import pprint

schema = WeatherRequest.model_json_schema()
pprint(schema)


{'properties': {'city': {'description': "City name, e.g., 'San Francisco'",
                         'title': 'City',
                         'type': 'string'},
                'unit': {'default': 'celsius',
                         'description': "Temperature unit: 'celsius' or "
                                        "'fahrenheit'",
                         'title': 'Unit',
                         'type': 'string'}},
 'required': ['city'],
 'title': 'WeatherRequest',
 'type': 'object'}


### Step 3: Invoke the tool via LLM
We’ll simulate a tool-augmented chat: the model chooses tool + args, returns structured output.


In [6]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_classic.agents import AgentExecutor, create_tool_calling_agent

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Use tools when needed."),
    ("human", "What's the temperature in {city} in {unit}?"),
    MessagesPlaceholder(variable_name="agent_scratchpad")
])

agent = create_tool_calling_agent(llm, tools=[get_weather], prompt=prompt)
executor = AgentExecutor(agent=agent, tools=[get_weather], verbose=True)

print(executor.invoke({"city": "San Francisco", "unit": "celsius"})["output"])




> Entering new AgentExecutor chain...

Invoking: `get_weather` with `{'city': 'San Francisco', 'unit': 'celsius'}`


{"city": "San Francisco", "temp": 18, "unit": "C"}


> Finished chain.
{"city": "San Francisco", "temp": 18, "unit": "C"}
